# A · 좌은진 — DeBERTa-v3-base + 주제 적대 학습 (A-5, A-6)

`eunjin` 브랜치. **Accelerator GPU T4, Internet On.** RF는 `A_rf_kaggle.ipynb`.

오래 걸리므로(약 20분) 결과를 남기려면 **Save Version → Save & Run All**로 실행하거나, 끝난 직후 4번에서 파일을 내려받는다.

## 1. 저장소·환경

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging
!git clone -b eunjin https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline

import os
assert os.path.exists('experiments/A_deberta.py'), 'eunjin 브랜치가 push 됐는지 확인'

import platform, sys, numpy, pandas, sklearn, scipy
print(platform.platform(), '| Python', sys.version.split()[0])
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| sklearn', sklearn.__version__, '| scipy', scipy.__version__)

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 기본형(λ=0) vs 주제 적대 학습(λ=0.1), 최대 10에폭, seed 42

In [ ]:
!python -u -W ignore experiments/A_deberta.py --epochs 10 --patience 3 --out-dir results/course/A/deberta_ep10 --pred-dir preds/deberta_ep10 --ckpt-dir artifacts/ckpt/A_deberta_ep10

## 3. 결과

In [ ]:
import pandas as pd
from IPython.display import Image, display as d
pd.set_option('display.width', 220)
B = 'results/course/A/deberta_ep10/'

res = pd.read_csv(B + 'val_results.csv')
cols = ['run_id', 'lambda', 'best_epoch', 'model_BA', 'macro_f1', 'company_BA', 'sibling_acc',
        'probe_train_topic_acc', 'probe_val_topic_acc', 'fit_seconds', 'peak_gpu_mem_gb']
display(res[cols].round(4))

### 3-1. 에폭별 val BA, 작성자 BA vs 주제 진단

In [ ]:
hist = pd.read_csv(B + 'history.csv')
display(hist.pivot(index='epoch', columns='lambda', values='val_BA').round(4))
d(Image(B + 'fig_lambda_comparison.png'))

## 4. 결과 저장

In [ ]:
!zip -qr /kaggle/working/A_deberta_results.zip results/course/A/deberta_ep10 preds/deberta_ep10
!ls -lh /kaggle/working/A_deberta_results.zip
!ls -lh artifacts/ckpt/A_deberta_ep10

체크포인트(각 약 700MB)는 저장소에 올리지 않는다. 필요하면 아래 링크로 내려받아 보관한다.

In [ ]:
from IPython.display import FileLink
for f in ['A_deberta_lam0_seed42.pt', 'A_deberta_lam0.1_seed42.pt']:
    d(FileLink(f'artifacts/ckpt/A_deberta_ep10/{f}'))

## 5. test (팀 합의 후에만)

2026-11-18 회의: test는 각자 채점하지 않는다. 팀 결정에 따라 실행한다. 저장된 체크포인트가 필요하다.

In [ ]:
# !python -u -W ignore experiments/A_deberta.py --final --out-dir results/course/A/deberta_ep10 --pred-dir preds/deberta_ep10 --ckpt-dir artifacts/ckpt/A_deberta_ep10